# Contrôle des indicateurs par IRIS (étapes 4 à 6)

Entrées : `data/processed/indicateurs_iris.parquet` (script 07) et `data/exports/carte.gpkg` (script 08).

In [1]:
import geopandas as gpd, pandas as pd
pd.set_option('display.max_columns', 50)
i = gpd.read_parquet('../data/processed/indicateurs_iris.parquet')
pc = i[i.code_departement != '75']
print(i.shape, i.crs.to_string(), '| période :', i.periode_24m.iloc[0])

(2752, 43) EPSG:2154 | période : 2024-01-01 / 2025-12-31


## 1. Couverture des indicateurs (petite couronne)

Une médiane de prix n'est calculée qu'à partir de 5 ventes.

In [2]:
cols = ['revenu_median', 'prix_m2_median_ancien_24m', 'prix_m2_median_vefa_24m', 'evol_prix_m2_ancien',
        'dist_station_actuelle_m', 'part_surface_qpv']
pc.groupby('type_iris')[cols].apply(lambda d: d.notna().mean().round(2))

,revenu_median,prix_m2_median_ancien_24m,prix_m2_median_vefa_24m,evol_prix_m2_ancien,dist_station_actuelle_m,part_surface_qpv
type_iris,,,,,,
A,0.20,0.43,0.13,0.30,1.0,0.0
D,0.25,0.25,0.00,0.25,1.0,0.0
H,0.96,0.86,0.12,0.85,1.0,0.0
Z,1.00,0.86,0.00,0.86,1.0,0.0


In [3]:
pc.groupby('code_departement')[['n_ventes_ancien_24m', 'n_ventes_vefa_24m']].describe().round(0).T

code_departement              92     93     94
n_ventes_ancien_24m count  616.0  614.0  530.0
                    mean    46.0   25.0   35.0
                    std     31.0   27.0   27.0
                    min      0.0    0.0    0.0
                    25%     22.0    5.0   12.0
                    50%     44.0   16.0   29.0
                    75%     68.0   36.0   50.0
                    max    171.0  247.0  136.0
n_ventes_vefa_24m   count  616.0  614.0  530.0
                    mean     3.0    2.0    3.0
                    std     12.0   10.0   10.0
                    min      0.0    0.0    0.0
                    25%      0.0    0.0    0.0
                    50%      0.0    0.0    0.0
                    75%      0.0    0.0    0.0
                    max    155.0  116.0  147.0

## 2. Distributions

In [4]:
pc.groupby('code_departement')[['prix_m2_median_ancien_24m', 'prix_m2_median_vefa_24m', 'evol_prix_m2_ancien',
    'revenu_median', 'dist_station_actuelle_m', 'dist_metro_futur_m']].median().round(3)

,prix_m2_median_ancien_24m,prix_m2_median_vefa_24m,evol_prix_m2_ancien,revenu_median,dist_station_actuelle_m,dist_metro_futur_m
code_departement,,,,,,
92,5915.376,7417.643,-0.085,31880.0,471.0,1415.5
93,3455.757,4800.000,-0.069,18770.0,639.0,1730.0
94,4301.540,5338.983,-0.081,24655.0,649.0,1995.0


In [5]:
pc['dist_station_actuelle_m'].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(0)

count    1760.0
mean      698.0
std       540.0
min        17.0
25%       346.0
50%       561.0
75%       876.0
90%      1337.0
max      5604.0
Name: dist_station_actuelle_m, dtype: float64

## 3. Cohérence : prix/m² médian par IRIS vs revenu médian

In [6]:
pc[['prix_m2_median_ancien_24m', 'revenu_median']].corr(method='spearman').round(2)

,prix_m2_median_ancien_24m,revenu_median
prix_m2_median_ancien_24m,1.00,0.69
revenu_median,0.69,1.00


## 4. Sélection par défaut (script 08)

Paramètres dans la colonne `parametres`.

In [7]:
e = gpd.read_file('../data/exports/carte.gpkg', layer='iris_indicateurs')
e = e[e.code_departement != '75']
print(e.parametres.iloc[0])
print('IRIS sélectionnés :', int(e.selection.sum()), '/', len(e))
(e[e.selection].groupby(['code_departement', 'nom_commune'])
   .agg(iris=('code_iris', 'size'), prix_m2_ancien=('prix_m2_median_ancien_24m', 'median'),
        revenu=('revenu_median', 'median'), dist_station=('dist_station_actuelle_m', 'median'))
   .sort_values('iris', ascending=False).round(0).head(25))

budget ancien 210000 €, VEFA 290000 €, 45 m², station <= 800 m, revenu >= 0 €
IRIS sélectionnés : 475 / 1760


iris  prix_m2_ancien   revenu  \
code_departement nom_commune                                            
94               Vitry-sur-Seine          29          3737.0  19220.0   
93               Saint-Denis              26          3675.0  17120.0   
                 Aubervilliers            24          3768.0  15470.0   
94               Créteil                  19          3235.0  20575.0   
93               Épinay-sur-Seine         14          2810.0  17480.0   
94               Choisy-le-Roi            13          3571.0  20720.0   
93               Noisy-le-Sec             12          3487.0  18685.0   
                 Bondy                    10          2873.0  18520.0   
                 Sevran                   10          2144.0  17395.0   
94               Villejuif                10          4181.0  20515.0   
93               La Courneuve             10          2816.0  15285.0   
                 Drancy                    9          3179.0  18925.0   
                 Rosny-sous-Bois           9          3864.0  19535.0   
                 Bobigny                   9          2831.0  16720.0   
                 Livry-Gargan              9          3169.0  22040.0   
92               Antony                    8          3992.0  29255.0   
93               Aulnay-sous-Bois          8          3693.0  23740.0   
                 Gagny                     8          3197.0  25690.0   
                 Villemomble               8          3559.0  27205.0   
                 Noisy-le-Grand            8          2933.0  19060.0   
92               Gennevilliers             8          4276.0  19240.0   
93               Pierrefitte-sur-Seine     8          2728.0  16310.0   
                 Stains                    7          2800.0  16890.0   
94               Champigny-sur-Marne       7          3359.0  23160.0   
                 Orly                      7          3384.0  17890.0   

                                        dist_station  
code_departement nom_commune                          
94               Vitry-sur-Seine               444.0  
93               Saint-Denis                   276.0  
                 Aubervilliers                 477.0  
94               Créteil                       500.0  
93               Épinay-sur-Seine              308.0  
94               Choisy-le-Roi                 261.0  
93               Noisy-le-Sec                  472.0  
                 Bondy                         410.0  
                 Sevran                        457.0  
94               Villejuif                     394.0  
93               La Courneuve                  256.0  
                 Drancy                        554.0  
                 Rosny-sous-Bois               390.0  
                 Bobigny                       291.0  
                 Livry-Gargan                  337.0  
92               Antony                        450.0  
93               Aulnay-sous-Bois              635.0  
                 Gagny                         598.0  
                 Villemomble                   503.0  
                 Noisy-le-Grand                461.0  
92               Gennevilliers                 276.0  
93               Pierrefitte-sur-Seine         290.0  
                 Stains                        683.0  
94               Champigny-sur-Marne           614.0  
                 Orly                          379.0